In [3]:
import pandas as pd
import numpy as np
import joblib
import kagglehub
import os

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef
)

# Load dataset
# Original download path (read-only)
kaggle_download_root = kagglehub.dataset_download('nelgiriyewithana/credit-card-fraud-detection-dataset-2023')
print(f"Kaggle dataset downloaded to: {kaggle_download_root}")
print('Data source import complete.')

file_path = os.path.join(kaggle_download_root, "creditcard_2023.csv")
data = pd.read_csv(file_path, sep=',')

# Correctly separate features (X) and target (y) from the DataFrame
X = data.drop(['id', 'Class'], axis=1) # Drop 'id' and 'Class' to get features
y = data['Class'] # 'Class' is the target variable

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create 'model' directory if it doesn't exist
if not os.path.exists("model"):
    os.makedirs("model")

joblib.dump(scaler, "model/scaler.pkl")

# Models
models = {
    "logistic": LogisticRegression(max_iter=500),
    "decision_tree": DecisionTreeClassifier(),
    "naive_bayes": GaussianNB(),
    "random_forest": RandomForestClassifier(n_estimators=50),
    "xgboost": XGBClassifier(n_estimators=50, eval_metric="logloss")
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "AUC": roc_auc_score(y_test, probs),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
        "MCC": matthews_corrcoef(y_test, preds)
    }

    results.append(metrics)
    joblib.dump(model, f"model/{name}.pkl", compress=5)
    joblib.dump(list(X.columns), "model/feature_names.pkl")


X_knn, y_knn = resample(
    X_train,
    y_train,
    n_samples=1500,
    random_state=42,
    stratify=y_train
)

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_knn, y_knn)

preds = knn_model.predict(X_test)
probs = knn_model.predict_proba(X_test)[:, 1]

results.append({
    "Model": "knn",
    "Accuracy": accuracy_score(y_test, preds),
    "AUC": roc_auc_score(y_test, probs),
    "Precision": precision_score(y_test, preds),
    "Recall": recall_score(y_test, preds),
    "F1": f1_score(y_test, preds),
    "MCC": matthews_corrcoef(y_test, preds)
})

joblib.dump(knn_model, "model/knn.pkl", compress=3)

comparison_df = pd.DataFrame(results)

comparison_df = comparison_df[
    ["Model", "Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
]

print("\nModel Comparison Table:")
print(comparison_df)

comparison_df.to_csv("model/comparison_model_metrics.csv", index=False)

df = pd.DataFrame(X_test)
df["target"] = y_test.values
print(df)
df.to_csv("model/model_metrics.csv", index=False)


Using Colab cache for faster access to the 'credit-card-fraud-detection-dataset-2023' dataset.
Kaggle dataset downloaded to: /kaggle/input/credit-card-fraud-detection-dataset-2023
Data source import complete.

Model Comparison Table:
           Model  Accuracy       AUC  Precision    Recall        F1       MCC
0       logistic  0.965215  0.993396   0.977125  0.952875  0.964847  0.930720
1  decision_tree  0.998004  0.998002   0.996970  0.999052  0.998010  0.996010
2    naive_bayes  0.918040  0.974501   0.975286  0.858151  0.912977  0.842236
3  random_forest  0.999886  0.999981   0.999772  1.000000  0.999886  0.999771
4        xgboost  0.999129  0.999968   0.998283  0.999982  0.999132  0.998260
5            knn  0.954795  0.987353   0.962523  0.946627  0.954509  0.909717
               0         1         2         3         4         5         6  \
0       0.420391 -0.069347 -0.570505  0.192687 -0.010419  0.425124 -0.355231   
1      -0.239511  0.252434 -0.375572  0.153965 -0.105935 -0.